<a href="https://colab.research.google.com/github/harnepal-hub/ai-trading-bot/blob/main/AI_trading_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ccxt pandas pandas-ta

In [9]:
import requests
import pandas as pd
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# --- CONFIGURATION ---
PAIRS = ["B-BTC_USDT", "B-ETH_USDT", "B-SOL_USDT"]
INITIAL_CAPITAL = 1200.00 # Base capital to calculate 1% risk per trade

def fetch_history(pair, interval, limit=1000):
    url = f"https://public.coindcx.com/market_data/candles?pair={pair}&interval={interval}&limit={limit}"
    try:
        response = requests.get(url, timeout=10)
        data = response.json()

        if isinstance(data, dict):
            print(f"⚠️ Exchange Warning for {interval}: {data}")
            return pd.DataFrame()

        df = pd.DataFrame(data)
        df = df.sort_values(by='time').reset_index(drop=True)
        df['datetime'] = pd.to_datetime(df['time'], unit='ms')
        df.set_index(pd.DatetimeIndex(df["datetime"]), inplace=True)
        for col in ['open', 'high', 'low', 'close', 'volume']:
            df[col] = df[col].astype(float)
        return df
    except Exception as e:
        print(f"⚠️ Network error: {e}")
        return pd.DataFrame()

print("🚀 INITIALIZING MULTI-PAIR BACKTEST ENGINE...")
print(f"Starting Capital: ${INITIAL_CAPITAL:,.2f} | Risk Per Trade: 1.0%\n")

# Global trackers for the final report
grand_won = 0
grand_lost = 0
grand_be = 0
total_net_profit = 0

for pair in PAIRS:
    print(f"==========================================")
    print(f"🔍 SCANNING: {pair}")
    print(f"==========================================")

    print(f"📥 Fetching 1-Hour Trend Data for {pair}...")
    df_regime = fetch_history(pair, "1h", 1000)
    time.sleep(2) # Prevent API rate limits

    print(f"📥 Fetching 15-Minute Execution Data for {pair}...")
    df_exec = fetch_history(pair, "15m", 1000)
    time.sleep(2)

    if df_regime.empty or df_exec.empty:
        print(f"❌ Skipping {pair} due to network error.\n")
        continue

    # ==========================================
    # GATE 1: 1-Hour Regime & Strength
    # ==========================================
    df_regime['EMA_20'] = df_regime['close'].ewm(span=20, adjust=False).mean()
    df_regime['EMA_50'] = df_regime['close'].ewm(span=50, adjust=False).mean()
    df_regime['ema_spread_pct'] = (df_regime['EMA_20'] - df_regime['EMA_50']).abs() / df_regime['close'] * 100

    df_regime['bull_regime'] = (df_regime['EMA_20'] > df_regime['EMA_50']) & (df_regime['ema_spread_pct'] > 0.15)
    df_regime['bear_regime'] = (df_regime['EMA_20'] < df_regime['EMA_50']) & (df_regime['ema_spread_pct'] > 0.15)

    df_regime['bull_regime_closed'] = df_regime['bull_regime'].shift(1)
    df_regime['bear_regime_closed'] = df_regime['bear_regime'].shift(1)

    df_exec['1h_bull'] = df_regime['bull_regime_closed'].reindex(df_exec.index, method='ffill')
    df_exec['1h_bear'] = df_regime['bear_regime_closed'].reindex(df_exec.index, method='ffill')

    # ==========================================
    # GATE 2: 15-Minute Pullback Indicators
    # ==========================================
    df_exec['SMA_20'] = df_exec['close'].rolling(window=20).mean()
    df_exec['STD_20'] = df_exec['close'].rolling(window=20).std()
    df_exec['BBL_20_2.0'] = df_exec['SMA_20'] - (df_exec['STD_20'] * 2)
    df_exec['BBU_20_2.0'] = df_exec['SMA_20'] + (df_exec['STD_20'] * 2)

    high_low = df_exec['high'] - df_exec['low']
    high_close = (df_exec['high'] - df_exec['close'].shift()).abs()
    low_close = (df_exec['low'] - df_exec['close'].shift()).abs()
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    df_exec['ATRr_14'] = tr.rolling(window=14).mean()
    df_exec.dropna(inplace=True)

    # ==========================================
    # LOCAL SIMULATION ENGINE
    # ==========================================
    # Reset local variables for this specific pair
    balance = INITIAL_CAPITAL
    position = None
    entry_price = 0
    sl = 0
    tp = 0
    position_size = 0
    breakeven_trigger = 0
    stop_moved_to_be = False

    trades_won = 0
    trades_lost = 0
    trades_breakeven = 0

    for index, current in df_exec.iterrows():
        bb_lower = current['BBL_20_2.0']
        bb_upper = current['BBU_20_2.0']

        if position is not None:
            if position == 'LONG':
                if not stop_moved_to_be and current['high'] >= breakeven_trigger:
                    sl = entry_price
                    stop_moved_to_be = True

                if current['low'] <= sl:
                    if stop_moved_to_be:
                        trades_breakeven += 1
                    else:
                        balance -= (entry_price - sl) * position_size
                        trades_lost += 1
                    position = None
                elif current['high'] >= tp:
                    balance += (tp - entry_price) * position_size
                    trades_won += 1
                    position = None

            elif position == 'SHORT':
                if not stop_moved_to_be and current['low'] <= breakeven_trigger:
                    sl = entry_price
                    stop_moved_to_be = True

                if current['high'] >= sl:
                    if stop_moved_to_be:
                        trades_breakeven += 1
                    else:
                        balance -= (sl - entry_price) * position_size
                        trades_lost += 1
                    position = None
                elif current['low'] <= tp:
                    balance += (entry_price - tp) * position_size
                    trades_won += 1
                    position = None
            continue

        close = current['close']
        atr = current['ATRr_14']

        if current['1h_bull'] == True:
            if close <= bb_lower:
                position = 'LONG'
                entry_price = close
                sl = entry_price - (atr * 2.0)
                tp = entry_price + (atr * 4.0)
                breakeven_trigger = entry_price + (atr * 2.0)
                position_size = (balance * 0.01) / (entry_price - sl)
                stop_moved_to_be = False

        elif current['1h_bear'] == True:
            if close >= bb_upper:
                position = 'SHORT'
                entry_price = close
                sl = entry_price + (atr * 2.0)
                tp = entry_price - (atr * 4.0)
                breakeven_trigger = entry_price - (atr * 2.0)
                position_size = (balance * 0.01) / (sl - entry_price)
                stop_moved_to_be = False

    # Pair Results
    pair_profit = balance - INITIAL_CAPITAL
    total_net_profit += pair_profit
    grand_won += trades_won
    grand_lost += trades_lost
    grand_be += trades_breakeven

    print(f"✅ {pair} Complete -> W: {trades_won} | L: {trades_lost} | BE: {trades_breakeven} | Net: ${pair_profit:,.2f}\n")

# ==========================================
# GRAND TOTAL REPORT
# ==========================================
grand_total_trades = grand_won + grand_lost + grand_be
overall_win_rate = (grand_won / grand_total_trades * 100) if grand_total_trades > 0 else 0

print("==========================================")
print("🏆 MULTI-PAIR PORTFOLIO RESULTS (Last ~10.4 Days)")
print("==========================================")
print(f"Total Trades Taken: {grand_total_trades}")
print(f"Total Won:          {grand_won}")
print(f"Total Lost:         {grand_lost}")
print(f"Total Breakeven:    {grand_be}")
print(f"True Win Rate:      {overall_win_rate:.1f}%")
print(f"Combined Profit:    ${total_net_profit:,.2f} USDT")
print("==========================================")

🚀 INITIALIZING MULTI-PAIR BACKTEST ENGINE...
Starting Capital: $1,200.00 | Risk Per Trade: 1.0%

🔍 SCANNING: B-BTC_USDT
📥 Fetching 1-Hour Trend Data for B-BTC_USDT...
📥 Fetching 15-Minute Execution Data for B-BTC_USDT...
✅ B-BTC_USDT Complete -> W: 3 | L: 2 | BE: 5 | Net: $48.11

🔍 SCANNING: B-ETH_USDT
📥 Fetching 1-Hour Trend Data for B-ETH_USDT...
📥 Fetching 15-Minute Execution Data for B-ETH_USDT...
✅ B-ETH_USDT Complete -> W: 5 | L: 5 | BE: 4 | Net: $59.96

🔍 SCANNING: B-SOL_USDT
📥 Fetching 1-Hour Trend Data for B-SOL_USDT...
📥 Fetching 15-Minute Execution Data for B-SOL_USDT...
✅ B-SOL_USDT Complete -> W: 3 | L: 7 | BE: 6 | Net: $-13.06

🏆 MULTI-PAIR PORTFOLIO RESULTS (Last ~10.4 Days)
Total Trades Taken: 40
Total Won:          11
Total Lost:         14
Total Breakeven:    15
True Win Rate:      27.5%
Combined Profit:    $95.01 USDT
